# **Evaluasi 3 RSBP: Neo4j dan Google Colab**
---

**Kelompok 13** — Silsilah Kasultanan Ngayogyakarta Hadiningrat

| Nama | NRP |
|---|---|
| Gilbran Mahdavikia Raja | 5025241134 |
| Nabila Shafa Rahayu | 5025241150 |
| Rahma Sakinah | 5025241222 |

Di tugas ini, kami menggunakan data silsilah dari Evaluasi 2. Grafnya kami buat di Neo4j, lalu kami jalankan query Cypher lewat Google Colab.

**Data yang digunakan:**
- 47 node `Person`.
- Relasi `berpasanganDengan` dan `memilikiAnak`.
- `nama` digunakan sebagai identitas orang. Properti lainnya adalah `jenisKelamin`, `tahunLahir`, `tahunPerkawinan`, dan `penguasa`.
- `tahunPerkawinan` disimpan pada node istri sesuai data yang tersedia di Evaluasi 2.

## **1. Persiapan**
---

Kami menyiapkan library Python dan menghubungkan notebook ke database Neo4j yang berisi silsilah keluarga.

In [1]:
# Library untuk menjalankan Cypher dari Python
!pip -q install py2neo


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from py2neo import Graph

pd.set_option('display.max_colwidth', None)

### **Koneksi ke Neo4j**
---

Kami menghubungkan Google Colab ke Jupyter yang berjalan di laptop menggunakan Local Runtime. Dengan cara ini, kami bisa mengakses database Neo4j secara langsung melalui `bolt://localhost:7687` tanpa menggunakan ngrok.

Password Neo4j dimasukkan saat kode dijalankan agar tidak tersimpan di notebook.

In [ ]:
from getpass import getpass
from py2neo import Graph

uri = "neo4j://localhost:7687"
username = "neo4j"
password = "12345678"

graph = Graph(uri, auth=(username, password))

def q(cypher, **params):
    # Biar hasil query langsung tampil sebagai tabel
    return graph.run(cypher, **params).to_data_frame()

### **Cek isi graf**
---

Kami mengecek jumlah node dan relasi untuk memastikan data yang masuk sesuai dengan silsilah Evaluasi 2. Hasil yang diharapkan adalah **47 orang, 21 relasi pasangan, dan 51 relasi orang tua–anak**.

In [4]:
display(q("MATCH (n:Person) RETURN count(n) AS jumlahPerson"))
display(q("MATCH ()-[r]->() RETURN type(r) AS relasi, count(r) AS jumlah ORDER BY relasi"))

,jumlahPerson
0,47


,relasi,jumlah
0,berpasanganDengan,21
1,memilikiAnak,51


## **2. Menjawab Soal a–d Evaluasi 2**
---

### **Soal a**
---

**Tampilkan nama penguasa, jenis kelamin penguasa, dan nama ayah, jika ayahnya pernah menjadi penguasa sebelumnya.**

Kami mencari hubungan ayah dan anak melalui `memilikiAnak`. Setelah itu, kami memilih pasangan yang sama-sama tercatat sebagai penguasa kerajaan. Pada data silsilah ini, ayah yang menjadi penguasa berada pada generasi sebelumnya.

In [5]:
query_a = """
MATCH (ayah:Person {jenisKelamin: 'laki-laki'})-[:memilikiAnak]->(p:Person)
WHERE p.penguasa IS NOT NULL
  AND ayah.penguasa = p.penguasa
RETURN p.nama AS namaPenguasa,
       p.jenisKelamin AS jenisKelamin,
       ayah.nama AS namaAyah
ORDER BY p.tahunLahir, p.nama
"""
q(query_a)

,namaPenguasa,jenisKelamin,namaAyah
0,Sri Sultan Hamengku Buwono II,laki-laki,Sri Sultan Hamengku Buwono I
1,Sri Sultan Hamengku Buwono III,laki-laki,Sri Sultan Hamengku Buwono II
2,Sri Sultan Hamengku Buwono IV,laki-laki,Sri Sultan Hamengku Buwono III
3,Sri Sultan Hamengku Buwono V,laki-laki,Sri Sultan Hamengku Buwono IV
4,Sri Sultan Hamengku Buwono VI,laki-laki,Sri Sultan Hamengku Buwono IV
5,Sri Sultan Hamengku Buwono VII,laki-laki,Sri Sultan Hamengku Buwono VI
6,Sri Sultan Hamengku Buwono VIII,laki-laki,Sri Sultan Hamengku Buwono VII
7,Sri Sultan Hamengku Buwono IX,laki-laki,Sri Sultan Hamengku Buwono VIII
8,Sri Sultan Hamengku Bawono Ka 10,laki-laki,Sri Sultan Hamengku Buwono IX


HB I tidak muncul karena ayahnya, Sunan Amangkurat IV, tidak tercatat sebagai penguasa Kasultanan Yogyakarta pada data ini. HB V dan HB VI muncul dengan ayah yang sama, yaitu HB IV.

### **Data pernikahan pertama untuk Soal b**
---

Di Evaluasi 2, urutan pernikahan disimpan pada ontologi. Namun, di Neo4j kami hanya memakai properti yang diminta pada instruksi tugas.

Karena itu, **data urutan pernikahan kami simpan sementara sebagai daftar Python**, bukan sebagai properti baru di Neo4j. Dari daftar tersebut, kami ambil pernikahan dengan urutan 1 untuk dipakai sebagai parameter query Cypher pada Soal b.

Data berikut tetap mengacu pada urutan pernikahan di Evaluasi 2.

In [6]:
data_urutan = [
    {
        "suami": "Sri Sultan Hamengku Buwono IX",
        "istri": "KRAy Pintakapurnama",
        "urutan": 1
    },
    {
        "suami": "Sri Sultan Hamengku Buwono IX",
        "istri": "KRAy Windyaningrum",
        "urutan": 2
    },
    {
        "suami": "Sri Sultan Hamengku Buwono I",
        "istri": "GKR Kadipaten",
        "urutan": 2
    },
    {
        "suami": "Sri Sultan Hamengku Buwono VIII",
        "istri": "Kanjeng Ratu Alit",
        "urutan": 5
    },
    {
        "suami": "Sri Sultan Hamengku Buwono VIII",
        "istri": "Raden Ayu Siti Katina",
        "urutan": 1
    },
    {
        "suami": "Sri Sultan Hamengku Buwono VI",
        "istri": "GKR Hamengku Buwono",
        "urutan": 1
    },
    {
        "suami": "Sri Sultan Hamengku Buwono VI",
        "istri": "GKR Sultan",
        "urutan": 2
    },
    {
        "suami": "Sri Sultan Hamengku Buwono V",
        "istri": "GKR Kencono (permaisuri I HB V)",
        "urutan": 1
    },
    {
        "suami": "Sri Sultan Hamengku Buwono V",
        "istri": "GKR Sekar Kedhaton",
        "urutan": 2
    },
    {
        "suami": "Sri Sultan Hamengku Bawono Ka 10",
        "istri": "GKR Hemas",
        "urutan": 1
    },
    {
        "suami": "KPH Notonegoro",
        "istri": "GKR Hayu",
        "urutan": 1
    },
    {
        "suami": "KPH Purbodiningrat",
        "istri": "GKR Maduretno",
        "urutan": 1
    },
    {
        "suami": "KPH Wironegoro",
        "istri": "GKR Mangkubumi",
        "urutan": 1
    },
    {
        "suami": "KPH Yudanegara",
        "istri": "GKR Bendara",
        "urutan": 1
    }
]

# Hanya ambil pasangan yang tercatat sebagai pernikahan pertama
pernikahan_pertama = [d for d in data_urutan if d["urutan"] == 1]

print("Jumlah pasangan pernikahan pertama:", len(pernikahan_pertama))

Jumlah pasangan pernikahan pertama: 9


### **Soal b**
---

**Tampilkan nama anak, nama ayah, nama ibu, dan tahun lahir kedua orang tua dari pernikahan pertama. Jika tahun lahir tidak ada, data anak tetap ditampilkan.**

Kami mencocokkan pasangan dari daftar pernikahan pertama dengan relasi `berpasanganDengan` di Neo4j. Setelah itu, kami mencari anak yang memiliki relasi `memilikiAnak` dari kedua orang tua tersebut. Kalau tahun lahir tidak tersedia, hasilnya akan kosong (`null`) tanpa menghilangkan data anak.

In [7]:
query_b = """
UNWIND $pernikahan AS d
MATCH (ayah:Person)-[:berpasanganDengan]->(ibu:Person)
WHERE ayah.nama = d.suami
  AND ibu.nama = d.istri
MATCH (ayah)-[:memilikiAnak]->(anak:Person)<-[:memilikiAnak]-(ibu)
RETURN anak.nama AS namaAnak,
       ayah.nama AS namaAyah,
       ayah.tahunLahir AS tahunLahirAyah,
       ibu.nama AS namaIbu,
       ibu.tahunLahir AS tahunLahirIbu
ORDER BY namaAyah, namaAnak
"""
q(query_b, pernikahan=pernikahan_pertama)

,namaAnak,namaAyah,tahunLahirAyah,namaIbu,tahunLahirIbu
0,RM Manteyyo Kuncoro Suryonegoro,KPH Notonegoro,NaN,GKR Hayu,1982
1,RAj Artie Ayya Fatimasari,KPH Wironegoro,NaN,GKR Mangkubumi,1972
2,RM Drasthya Wironegoro,KPH Wironegoro,NaN,GKR Mangkubumi,1972
3,RAj Nisaka Irdina Yudanegara,KPH Yudanegara,NaN,GKR Bendara,1986
4,RM Radityo Mandhala Yudanegara,KPH Yudanegara,NaN,GKR Bendara,1986
5,GKR Bendara,Sri Sultan Hamengku Bawono Ka 10,1946.0,GKR Hemas,1952
6,GKR Condrokirono,Sri Sultan Hamengku Bawono Ka 10,1946.0,GKR Hemas,1952
7,GKR Hayu,Sri Sultan Hamengku Bawono Ka 10,1946.0,GKR Hemas,1952
8,GKR Maduretno,Sri Sultan Hamengku Bawono Ka 10,1946.0,GKR Hemas,1952
9,GKR Mangkubumi,Sri Sultan Hamengku Bawono Ka 10,1946.0,GKR Hemas,1952


Dari data Evaluasi 2, beberapa anak tidak termasuk hasil karena mereka tercatat lahir dari pernikahan kedua atau setelahnya. Dengan cara ini, kami tetap bisa menggunakan informasi pernikahan pertama tanpa mengubah graf Neo4j.

### **Soal c**
---

**Tampilkan nama cucu dan tahun lahir dari penguasa generasi pertama, diurutkan berdasarkan tahun lahirnya.**

Kami mencari penguasa yang tidak memiliki orang tua dengan status penguasa kerajaan yang sama. Dari orang tersebut, kami mengikuti dua relasi `memilikiAnak` untuk menemukan cucunya.

In [8]:
query_c = """
MATCH (p:Person)
WHERE p.penguasa IS NOT NULL
  AND NOT EXISTS {
      MATCH (ortu:Person)-[:memilikiAnak]->(p)
      WHERE ortu.penguasa = p.penguasa
  }
MATCH (p)-[:memilikiAnak]->(:Person)-[:memilikiAnak]->(cucu:Person)
RETURN DISTINCT cucu.nama AS namaCucu,
       cucu.tahunLahir AS tahunLahir
ORDER BY CASE WHEN tahunLahir IS NULL THEN 0 ELSE 1 END,
         tahunLahir
"""
q(query_c)

,namaCucu,tahunLahir
0,GKR Hangger,NaN
1,Sri Sultan Hamengku Buwono III,1769.0


Dari silsilah yang kami gunakan, penguasa generasi pertama adalah HB I. Cucunya adalah GKR Hangger (tahun lahir tidak tercatat) dan HB III (lahir 1769). Data tanpa tahun lahir ditampilkan lebih dulu agar urutannya sama dengan hasil Evaluasi 2.

### **Soal d**
---

**Tampilkan pasangan menikah yang merupakan sepupu, yaitu pasangan yang ayahnya bersaudara.**

Kami mencari pasangan yang terhubung lewat `berpasanganDengan`. Lalu, kami mengecek apakah ayah mereka memiliki ayah yang sama. Syarat `ayahL <> ayahP` digunakan agar yang ditemukan adalah sepupu, bukan saudara kandung.

In [9]:
query_d = """
MATCH (kakek:Person)-[:memilikiAnak]->(ayahL:Person)
      -[:memilikiAnak]->(laki:Person),
      (kakek)-[:memilikiAnak]->(ayahP:Person)
      -[:memilikiAnak]->(perempuan:Person),
      (laki)-[:berpasanganDengan]-(perempuan)
WHERE kakek.jenisKelamin = 'laki-laki'
  AND ayahL.jenisKelamin = 'laki-laki'
  AND ayahP.jenisKelamin = 'laki-laki'
  AND laki.jenisKelamin = 'laki-laki'
  AND perempuan.jenisKelamin = 'perempuan'
  AND ayahL <> ayahP
RETURN DISTINCT laki.nama AS namaLakiLaki,
       perempuan.nama AS namaPerempuan,
       ayahL.nama AS ayahLakiLaki,
       ayahP.nama AS ayahPerempuan,
       kakek.nama AS kakekBersama
"""
q(query_d)

,namaLakiLaki,namaPerempuan,ayahLakiLaki,ayahPerempuan,kakekBersama
0,Sri Sultan Hamengku Buwono VIII,Raden Ayu Siti Katina,Sri Sultan Hamengku Buwono VII,GPAA Mangkubumi,Sri Sultan Hamengku Buwono VI
1,Sri Sultan Hamengku Buwono VIII,Kanjeng Ratu Alit,Sri Sultan Hamengku Buwono VII,GPAA Mangkubumi,Sri Sultan Hamengku Buwono VI


Dari silsilah, HB VIII menikah dengan dua putri GPAA Mangkubumi. Ayah HB VIII, yaitu HB VII, dan GPAA Mangkubumi sama-sama anak HB VI. Jadi, dua pernikahan tersebut termasuk pasangan sepupu sesuai kriteria soal.

## **3. Pertanyaan baru dengan Graph Data Science**
---

### **Pertanyaan**
---

**Siapa lima orang yang paling berperan sebagai penghubung dalam silsilah Kasultanan Yogyakarta?**

Untuk menjawabnya, kami memakai **Betweenness Centrality**. Nilai ini menunjukkan seberapa sering seseorang berada di jalur terpendek yang menghubungkan anggota keluarga lain.

Kami mengecek plugin GDS terlebih dahulu, membuat proyeksi graf dengan hubungan dua arah, lalu menghitung nilainya dan mengambil lima orang teratas.

In [11]:
# Memastikan plugin Graph Data Science sudah terpasang
q("RETURN gds.version() AS versiGDS")

,versiGDS
0,2026.10.2


In [12]:
# Jika ada proyeksi dari percobaan sebelumnya, kami hapus dulu
cek = q("CALL gds.graph.exists('silsilah') YIELD exists RETURN exists")
if bool(cek.loc[0, 'exists']):
    q("CALL gds.graph.drop('silsilah') YIELD graphName RETURN graphName")

# Memasukkan graf keluarga ke GDS
query_proyeksi = """
CALL gds.graph.project(
  'silsilah',
  'Person',
  {
    memilikiAnak:      {orientation: 'UNDIRECTED'},
    berpasanganDengan: {orientation: 'UNDIRECTED'}
  }
)
YIELD graphName, nodeCount, relationshipCount
RETURN graphName, nodeCount, relationshipCount
"""
q(query_proyeksi)

,graphName,nodeCount,relationshipCount
0,silsilah,47,144


In [13]:
query_gds = """
CALL gds.betweenness.stream('silsilah')
YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS orang, score
RETURN orang.nama AS nama,
       orang.penguasa IS NOT NULL AS pernahBerkuasa,
       round(score, 2) AS betweenness
ORDER BY betweenness DESC, nama
LIMIT 5
"""
q(query_gds)

,nama,pernahBerkuasa,betweenness
0,Sri Sultan Hamengku Buwono VI,True,531.75
1,Sri Sultan Hamengku Buwono IX,True,531.00
2,Sri Sultan Hamengku Bawono Ka 10,True,524.00
3,Sri Sultan Hamengku Buwono IV,True,466.00
4,Sri Sultan Hamengku Buwono III,True,376.00


### **Penjelasan hasil GDS**
---

Kami mengurutkan lima orang dengan nilai **betweenness** terbesar. Semakin besar nilainya, semakin sering orang tersebut menjadi penghubung di antara anggota keluarga lain dalam graf.

Pada silsilah ini, hubungan antargenerasi dan pernikahan yang menyambungkan dua cabang keluarga bisa membuat nilai seseorang lebih tinggi. Sebaliknya, orang yang hanya punya satu koneksi pasti mendapat nilai 0 karena tidak mungkin menjadi perantara antara dua orang lain.

*Hasil akhir mengikuti tabel yang muncul setelah query dijalankan.*

In [14]:
# Menghapus proyeksi GDS dari memori setelah selesai
q("CALL gds.graph.drop('silsilah') YIELD graphName RETURN graphName")

,graphName
0,silsilah
